# 08 — Matrix Computation（矩阵计算：GEMV & GEMM）

## 背景

矩阵乘法（GEMM）是深度学习中计算量最大的操作：线性层、注意力投影、前馈网络全部依赖 GEMM。现代 GPU 的 Tensor Core 专为矩阵乘提供了硬件加速（H100 SXM 上 fp16 达 1979 TFLOPS），但只有充分利用以下技术才能逼近峰值：

- **Tensor Core（MMA）**：每次处理 16×16 或更大的矩阵块，吞吐量是 CUDA Core 的 8–16×
- **共享内存（Shared Memory）**：片上 SRAM，带宽比 HBM 高 ~100×
- **软件流水线（Pipelining）**：让全局内存加载与 Tensor Core 计算重叠，隐藏内存延迟

## 问题定义

**GEMM**：$C[i,j] = \sum_k A[i,k] \times B[k,j]$，$A\in\mathbb{R}^{M\times K}$，$B\in\mathbb{R}^{K\times N}$

| 实现 | A/B 缓存 | K 循环 |
|------|---------|--------|
| Triton | 自动（寄存器/共享内存）| `for k in range(0,K,BK)` |
| TileLang 朴素 | `alloc_fragment`（寄存器）| `T.Serial` |
| TileLang 优化 | `alloc_shared`（共享内存）| `T.Pipelined(num_stages=3)` |

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
M, N, K = 4096, 4096, 4096
A = torch.randn(M, K, dtype=torch.float16, device="cuda")
B = torch.randn(K, N, dtype=torch.float16, device="cuda")

def ref_gemm(A, B): return torch.matmul(A, B)

C_ref = ref_gemm(A, B)
print(f"A: {A.shape} × B: {B.shape} → C: {C_ref.shape}")
print(f"理论 FLOPS: {2*M*N*K/1e12:.2f} TFLOPS·ms⁻¹ = {2*M*N*K:.2e} FLOPs")

## Triton 实现

Triton GEMM 是社区最广泛参考的高性能内核之一：
- 二维 Block 网格对应输出矩阵分块
- 内层 K 循环加载 A/B tile，调用 `tl.dot` 触发 Tensor Core MMA
- `allow_tf32=True` 启用 TF32 加速（在支持的 GPU 上）

In [ ]:
@triton.jit
def _matmul_kernel(A_ptr, B_ptr, C_ptr, M, N, K,
                   BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rows  = pid_m * BM + tl.arange(0, BM)   # [BM]
    cols  = pid_n * BN + tl.arange(0, BN)   # [BN]
    acc   = tl.zeros([BM, BN], dtype=tl.float32)
    for k in range(0, K, BK):
        k_range = k + tl.arange(0, BK)      # [BK]
        # Load A tile [BM, BK]
        a = tl.load(A_ptr + rows[:, None] * K + k_range[None, :],
                    mask=(rows[:, None] < M) & (k_range[None, :] < K), other=0.0)
        # Load B tile [BK, BN]
        b = tl.load(B_ptr + k_range[:, None] * N + cols[None, :],
                    mask=(k_range[:, None] < K) & (cols[None, :] < N), other=0.0)
        acc = acc + tl.dot(a, b, allow_tf32=True)
    c = acc.to(tl.float16)
    tl.store(C_ptr + rows[:, None] * N + cols[None, :], c,
             mask=(rows[:, None] < M) & (cols[None, :] < N))

BM_t, BN_t, BK_t = 128, 128, 64

def triton_gemm(A, B):
    M_, K_ = A.shape;  _, N_ = B.shape
    C = torch.empty(M_, N_, dtype=torch.float16, device=A.device)
    grid = (triton.cdiv(M_, BM_t), triton.cdiv(N_, BN_t))
    _matmul_kernel[grid](A, B, C, M_, N_, K_, BM=BM_t, BN=BN_t, BK=BK_t)
    return C

C_tri = triton_gemm(A, B)
ok = torch.allclose(C_ref.half(), C_tri.half(), atol=1.0)
print("Triton 正确性验证:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(C_ref - C_tri)).item():.4f}")

## TileLang 实现

展示两个版本：**朴素版**（`alloc_fragment` + `T.Serial`）和**优化版**（`alloc_shared` + `T.Pipelined`），直观展示优化效果。

In [ ]:
BM, BN, BK = 128, 128, 64

# 朴素版：寄存器缓冲 + 串行 K 循环
@tilelang.jit
def tl_gemm_naive(A, B, BLOCK_M: int, BLOCK_N: int, BLOCK_K: int):
    M_, N_, K_ = T.const("M, N, K")
    dtype, acc = T.float16, T.float32
    A: T.Tensor((M_, K_), dtype);  B: T.Tensor((K_, N_), dtype)
    C = T.empty((M_, N_), dtype)
    with T.Kernel(T.ceildiv(N_, BLOCK_N), T.ceildiv(M_, BLOCK_M), threads=128) as (pid_n, pid_m):
        A_l = T.alloc_fragment((BLOCK_M, BLOCK_K), dtype)
        B_l = T.alloc_fragment((BLOCK_K, BLOCK_N), dtype)
        C_l = T.alloc_fragment((BLOCK_M, BLOCK_N), acc)
        T.clear(C_l)
        for k in T.Serial(K_ // BLOCK_K):
            T.copy(A[pid_m * BLOCK_M, k * BLOCK_K], A_l)
            T.copy(B[k * BLOCK_K, pid_n * BLOCK_N], B_l)
            T.gemm(A_l, B_l, C_l)
        T.copy(C_l, C[pid_m * BLOCK_M, pid_n * BLOCK_N])
    return C

# 优化版：共享内存 + 软件流水线
@tilelang.jit
def tl_gemm_opt(A, B, BLOCK_M: int, BLOCK_N: int, BLOCK_K: int):
    M_, N_, K_ = T.const("M, N, K")
    dtype, acc = T.float16, T.float32
    A: T.Tensor((M_, K_), dtype);  B: T.Tensor((K_, N_), dtype)
    C = T.empty((M_, N_), dtype)
    with T.Kernel(T.ceildiv(N_, BLOCK_N), T.ceildiv(M_, BLOCK_M), threads=128) as (pid_n, pid_m):
        A_s = T.alloc_shared((BLOCK_M, BLOCK_K), dtype)   # 共享内存
        B_s = T.alloc_shared((BLOCK_K, BLOCK_N), dtype)
        C_l = T.alloc_fragment((BLOCK_M, BLOCK_N), acc)
        T.clear(C_l)
        for k in T.Pipelined(K_ // BLOCK_K, num_stages=3):  # 3 级流水
            T.copy(A[pid_m * BLOCK_M, k * BLOCK_K], A_s)
            T.copy(B[k * BLOCK_K, pid_n * BLOCK_N], B_s)
            T.gemm(A_s, B_s, C_l)
        T.copy(C_l, C[pid_m * BLOCK_M, pid_n * BLOCK_N])
    return C

hyp = {"M": M, "N": N, "K": K, "BLOCK_M": BM, "BLOCK_N": BN, "BLOCK_K": BK}
k_naive = tl_gemm_naive.compile(**hyp)
k_opt   = tl_gemm_opt.compile(**hyp)

def check(name, out):
    ok = torch.allclose(C_ref.half(), out.half(), atol=1.0)
    print(f"  {name:15s}: {'✅ PASSED' if ok else '❌ FAILED'}")

check("TileLang 朴素", k_naive(A.clone(), B.clone()))
check("TileLang 优化", k_opt(A.clone(), B.clone()))

## 性能对比

双图：延迟（ms）和 TFLOPS，直观展示各实现与理论峰值的差距。

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 100

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

flops = 2 * M * N * K

results = {
    "PyTorch\n(cuBLAS)": bench(ref_gemm,   [A, B]),
    "Triton":            bench(triton_gemm, [A, B]),
    "TileLang\n朴素":   bench(k_naive,    [A, B]),
    "TileLang\n优化":   bench(k_opt,      [A, B]),
}

for name, ms in results.items():
    tf = flops / ms / 1e9
    print(f"  {name.replace(chr(10),' '):20s}: {ms:.4f} ms  ({tf:.1f} TFLOPS)")

colors = ["#4C72B0", "#55A868", "#DD8452", "#C44E52"]
labels = list(results.keys())
ms_vals = list(results.values())
tf_vals = [flops / ms / 1e9 for ms in ms_vals]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
bars = ax.bar(labels, ms_vals, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, ms_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(ms_vals)*0.01,
            f"{val:.2f} ms", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("延迟 (ms)"); ax.set_title(f"GEMM 延迟  (M=N=K={M}, float16)")
ax.set_ylim(0, max(ms_vals)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bars = ax.bar(labels, tf_vals, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, tf_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(tf_vals)*0.01,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("计算吞吐 (TFLOPS)"); ax.set_title(f"GEMM 吞吐  (M=N=K={M}, float16)")
ax.set_ylim(0, max(tf_vals)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()